In [31]:
path2pic = 'reference_images'
path2triplet = 'triplet_dataset'
triplet_dataset = 'triplets_large_final_correctednc_correctedorder.csv'
triplet_index = 'unique_id.txt'
random_index = 'random_index.txt'
img_extension = '.jpg'

# Load and organize data

In [32]:
import os
root_dir = os.path.dirname(os.getcwd())

pic_dir = os.path.join(root_dir, path2pic)
triplet_data = os.path.join(root_dir, path2triplet, triplet_dataset)
triplet_index = os.path.join(root_dir, path2triplet, triplet_index)
random_index = os.path.join(root_dir, path2triplet, random_index)

## Get picture name

In [33]:
with open(triplet_index) as f:
    picture_names = [line.strip()+img_extension for line in f.readlines()]
picture_names[0:3]

['aardvark.jpg', 'abacus.jpg', 'accordion.jpg']

In [34]:
len(picture_names)

1854

## Get triplet data

### Extract random sample from whole dataset
cause the original dataset is too large

In [35]:
import pandas as pd

# read data
triplet = pd.read_csv(triplet_data, sep="\t")
triplet.head(5)

,image1,image2,image3,choice,RT,noise_ceiling,subject_id,HIT_nr,trial_nr,age,gender,date,time,dataset
0,1245,1050,494,3,14457,0,BMWLG5ZY79VLA,1,1,NaN,other,2018-02-05,09:31:16,1
1,888,1788,1250,3,5043,0,BMWLG5ZY79VLA,1,2,NaN,other,2018-02-05,09:31:16,1
2,1256,734,946,2,6605,0,BMWLG5ZY79VLA,1,3,NaN,other,2018-02-05,09:31:16,1
3,1069,1037,953,1,6177,0,BMWLG5ZY79VLA,1,4,NaN,other,2018-02-05,09:31:16,1
4,1693,803,963,3,2327,0,BMWLG5ZY79VLA,1,5,NaN,other,2018-02-05,09:31:16,1


In [36]:
len(triplet)

4699160

In [ ]:
# # Randomly select 100,000 data, cause the dataset is too large: 469,9160
# random_index_label = triplet.sample(n=100000).index
# # to make sure that all the images are included
# image_set = set() 
# for index in random_index_label:
#     img1 = triplet["image1"].iloc[index]
#     img2 = triplet["image2"].iloc[index]
#     img3 = triplet["image3"].iloc[index]
#     image_set.add(img1)
#     image_set.add(img2)
#     image_set.add(img3)
# print(random_index_label.unique())
# print(len(image_set), len(image_set)==len(picture_names))

Index([1765045, 2375681, 1499744, 4002372,  439985, 3695146, 2267043,  779848,
        258026, 1629022,
       ...
        842784, 1866350, 3194827, 1207780, 3936674, 1984865, 4256051, 3331945,
       3209071, 3468089],
      dtype='int64', length=100000)
1854 True


In [ ]:
# with open(random_index, mode = "w") as f:
#     for index in random_index_label:
#         f.write(str(index) + '\n')

In [40]:
with open(random_index) as f:
    index_selected = [line.strip() for line in f.readlines()]

In [42]:
len(index_selected)

100000

### Permutation of a, b, and c
In this project, each sample is defined as a triplet of a, b, and c, representing the items in odd-one-out judgment. They function as positional indices used to distinguish the three slots within a triplet, instead of a new set of stimuli. 
Changing the ordering of samples aims to avoid the model’s learning of positional biases, or treating the position itself as informative. To prevent this, we permute the order of a, b, and c during the data processing and exploratory data analysis. The permutations of the same triplet represent an equivalent set (e.g., (a, b, c), (b, a, c), (c, b, a)) and should yield the same interpretations.

In [56]:
triplet_sample = pd.DataFrame(triplet, index = index_selected)

In [59]:
# detect duplicate
duplicate_detect_set = set()
for index, row in triplet_sample.iterrows():
    duplicate_detect_set.add((row["image1"], row["image2"], row["image3"]))
print(len(duplicate_detect_set), len(duplicate_detect_set)==len(index_selected))

100000 True


In [ ]:
data_sample = []
for index, row in triplet_sample.iterrows():
    print(index)
    print(row)

0
image1                    1245
image2                    1050
image3                     494
choice                       3
RT                       14457
noise_ceiling                0
subject_id       BMWLG5ZY79VLA
HIT_nr                       1
trial_nr                     1
age                        NaN
gender                   other
date                2018-02-05
time                  09:31:16
dataset                      1
Name: 0, dtype: object
1
image1                     888
image2                    1788
image3                    1250
choice                       3
RT                        5043
noise_ceiling                0
subject_id       BMWLG5ZY79VLA
HIT_nr                       1
trial_nr                     2
age                        NaN
gender                   other
date                2018-02-05
time                  09:31:16
dataset                      1
Name: 1, dtype: object
2
image1                    1256
image2                     734
image3            

KeyboardInterrupt: 

## Convert pictures into pixels

In [39]:
import numpy as np
from PIL import Image

COMMON_SIZE = (224, 224)

def load_image_pixels(path, size=None):
    img = Image.open(path).convert("RGB")
    if size is not None:
        img = img.resize(size, Image.BILINEAR)
    arr = np.asarray(img, dtype=np.float32) / 255.0  # (H, W, 3)
    return arr

def image_to_vector(path, size=None):
    arr = load_image_pixels(path, size=size)
    return arr.reshape(-1)  # (H*W*3,)